# MedRAG Retriever Comparison for the Transplant Manual

## Project Purpose

This notebook documents an initial retrieval comparison for a retrieval-augmented generation system, also called a RAG system. A RAG system first retrieves relevant source text, then sends that text to a language model so the generated answer is grounded in a document. This design is appropriate for the heart transplant teaching manual because the application is intended to answer from the manual instead of general medical knowledge.

[MedRAG](https://github.com/gzxiong/MedRAG) is used as a reference point for medical RAG system design. MedRAG provides a framework for comparing medical retrievers, corpora, and language models. This notebook applies the same comparison idea to the project manual by testing four retriever families: BM25, Contriever, SPECTER, and MedCPT.

The heart transplant teaching manual remains the only knowledge source. This keeps the experiment aligned with the planned iOS application and the planned FastAPI backend, which are intended to return manual-grounded answers, lessons, and quizzes.

## Notebook Structure

This notebook follows a sequential experimental process.

1. First, it creates cleaned manual text and retrieval chunks.
2. Then, it builds multiple retrieval methods.
3. Then, it compares the retrievers on representative questions.
4. Then, it uses the retrieval findings to run generation tests in the planned backend response format.
5. Then, it prepares backend package files for later integration.
6. Finally, it verifies that the exported backend package can be reloaded.

## References

- MedRAG repository: [https://github.com/gzxiong/MedRAG](https://github.com/gzxiong/MedRAG)
- Original RAG paper by Lewis et al.: [https://arxiv.org/abs/2005.11401](https://arxiv.org/abs/2005.11401)
- FAISS documentation: [https://faiss.ai/](https://faiss.ai/)
- Groq documentation: [https://console.groq.com/docs/overview](https://console.groq.com/docs/overview)


In [1]:
# Install required packages for Colab.
# This notebook compares the four retriever families used by MedRAG: BM25, Contriever, SPECTER, and MedCPT.
# It keeps the heart transplant manual as the source corpus so the generated outputs stay grounded in the project document.

!pip install -q pypdf pandas numpy faiss-cpu rank_bm25 sentence-transformers transformers accelerate groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 5.7 MB/s eta 0:00:00


# Step 1: Environment Setup

This section installs and imports the libraries required for the experiment. The notebook uses PDF extraction tools, ranking libraries, transformer models, FAISS indexes, and the Groq API.

The code checks whether a GPU is available. A GPU can make dense retriever embedding faster, but the notebook can still run on CPU. In the saved run, the notebook used CPU.


In [2]:
# Import libraries.

import json
import os
import re
import time
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import torch
from google.colab import drive
from groq import Groq
from pypdf import PdfReader
from rank_bm25 import BM25Okapi
from transformers import AutoModel, AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cpu


# Step 2: Google Drive and File Paths

This section connects Colab to Google Drive. Google Drive is used because Colab runtime storage is temporary. Files saved to Drive remain available after the notebook session ends.

The project folder is `high_risk_project`. The manual PDF is stored in that folder. Generated files are written to `medrag_retriever_outputs` inside the same Drive folder.


In [3]:
# Mount Google Drive and define project paths.

drive.mount("/content/drive")

BASE_PATH = Path("/content/drive/MyDrive/high_risk_project")
PDF_PATH = BASE_PATH / "Heart Transplant_Post-transplant Teaching Manual_English_Dec 2024.pdf"

OUTPUT_PATH = BASE_PATH / "medrag_retriever_outputs"
DATA_PATH = OUTPUT_PATH / "data"
INDEX_PATH = OUTPUT_PATH / "indexes"
REPORT_PATH = OUTPUT_PATH / "reports"

for path in [OUTPUT_PATH, DATA_PATH, INDEX_PATH, REPORT_PATH]:
    path.mkdir(parents=True, exist_ok=True)

print("PDF exists:", PDF_PATH.exists())
print("Output path:", OUTPUT_PATH)

Mounted at /content/drive
PDF exists: True
Output path: /content/drive/MyDrive/high_risk_project/medrag_retriever_outputs


# Step 3: Manual Text Extraction and Cleaning

This section loads the heart transplant teaching manual from the PDF file. The PDF text is extracted page by page and combined into one text string.

The cleaning step removes common PDF artifacts. These include extra whitespace, page markers, repeated website footer text, and table-of-contents dot leaders. The goal is not to perfectly reconstruct the manual layout. The goal is to produce readable text that can be split into retrieval chunks.


In [ ]:
# Extract and clean text from the PDF manual.

def extract_pdf_text(pdf_path):
    reader = PdfReader(str(pdf_path))
    pages = []
    for page in reader.pages:
        text = page.extract_text()
        if text:
            pages.append(text)
    return "\n\n".join(pages)


def clean_text(text):
    text = text.replace("\u00a0", " ")
    text = re.sub(r"(\/\s*){5,}", " ", text)
    text = re.sub(r".*\.{5,}\s*\d+\s*", "", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"stanfordchildrens\.org.*?\n", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\|\s*\d{3,5}", "", text)
    return text.strip()


raw_text = extract_pdf_text(PDF_PATH)
cleaned_text = clean_text(raw_text)

(DATA_PATH / "cleaned_manual.txt").write_text(cleaned_text, encoding="utf-8")

print("Raw text length:", len(raw_text))
print("Cleaned text length:", len(cleaned_text))
print(cleaned_text[:1000])

Raw text length: 81793
Cleaned text length: 59953
P002100  
 
 
 
 
Heart Transplant Teaching Manual 
 
Your child is leaving the hospital after getting a new heart. This manual will help explain 
the extra care your child will need at home, especially during the first year after 
transplant. 
 
Living with a transplanted heart is a lifelong journey for your child and your whole family. We created this 
manual in partnership with others who have walked journey before you. Please use this manual to help keep 
your child’s new heart healthy through good planning, organization, and partnership with the transplant team 
at Stanford Medicine Children’s Health. We will be with you every step of the way. 
 
Table of Contents 
P002100  
 
 
 
 
P002100  
 
 
 
 
P002100  
 
 
 
 
P002100  
 
 
 
 
What to Watch Out for After Discharge 
Use this page to quickly find signs and symptoms to watch out for, and when to call the 
transplant team. See below for the safe range of vital signs for your c

# Step 4: Chunk Creation

This section splits the cleaned manual into smaller text chunks. A chunk is a short passage that can be retrieved and passed into the language model.

The notebook uses sentence-based chunking. This keeps full sentences together instead of cutting text in the middle of a sentence. It also keeps a small sentence overlap between neighboring chunks so that context is not lost at chunk boundaries.

The notebook also creates MedRAG-style snippets. These snippets include an ID, title, content, and combined contents field. This format supports later retriever experiments by keeping each passage in a structured document-like form.


In [ ]:
# Split the manual into sentence-based chunks.
# The output format includes both plain backend chunks and MedRAG-style snippets.

def split_sentences(text):
    text = re.sub(r"\s+", " ", text).strip()
    pattern = r"(?<=[.!?])\s+(?=[A-Z0-9])"
    sentences = re.split(pattern, text)
    return [sentence.strip() for sentence in sentences if sentence.strip()]


def chunk_text(text, max_chars=500, overlap_sentences=1):
    sentences = split_sentences(text)
    chunks = []
    current_chunk = []
    current_length = 0

    for sentence in sentences:
        sentence_length = len(sentence)
        if current_chunk and current_length + sentence_length > max_chars:
            chunks.append(" ".join(current_chunk).strip())
            current_chunk = current_chunk[-overlap_sentences:] if overlap_sentences > 0 else []
            current_length = sum(len(item) for item in current_chunk)

        current_chunk.append(sentence)
        current_length += sentence_length

    if current_chunk:
        chunks.append(" ".join(current_chunk).strip())

    return chunks


chunks = chunk_text(cleaned_text, max_chars=500, overlap_sentences=1)

snippets = []
for idx, chunk in enumerate(chunks):
    snippet = {
        "id": f"heart_transplant_manual_{idx:04d}",
        "title": "Heart Transplant Teaching Manual",
        "content": chunk,
        "contents": "Heart Transplant Teaching Manual.\n" + chunk,
    }
    snippets.append(snippet)

(DATA_PATH / "chunks.json").write_text(json.dumps(chunks, ensure_ascii=False, indent=2), encoding="utf-8")
(DATA_PATH / "medrag_style_snippets.json").write_text(json.dumps(snippets, ensure_ascii=False, indent=2), encoding="utf-8")

print("Total chunks:", len(chunks))
print(chunks[0][:800])

Total chunks: 175
P002100 Heart Transplant Teaching Manual Your child is leaving the hospital after getting a new heart. This manual will help explain the extra care your child will need at home, especially during the first year after transplant. Living with a transplanted heart is a lifelong journey for your child and your whole family. We created this manual in partnership with others who have walked journey before you.


# Step 5: Evaluation Questions

This section defines four test questions for retrieval evaluation. The questions are designed to test different kinds of information from the manual.

First, the temperature question tests whether a retriever finds a specific numeric threshold. Then, the transplant team question tests whether a retriever finds parent action instructions. The rejection question tests whether a retriever finds symptom and warning-sign information.

The fourth question tests natural user phrasing. This is important because the `/ask` endpoint may receive questions that do not use the same words as the manual. A parent may describe a child as feeling hot or acting different instead of asking directly about fever or rejection.

The notebook retrieves the top three chunks for each question. This keeps the evaluation small enough for manual review while still comparing all four retrievers.


In [8]:
# Define four evaluation questions.
# These test different retrieval needs: numeric threshold, parent action, warning signs, and natural user phrasing.

EVAL_QUERIES = [
    "What temperature is considered dangerous after transplant?",
    "When should I call the transplant team?",
    "What are signs of rejection after transplant?",
    "My child feels hot, is tired, and seems different than usual. What should I do?",
]

ASK_TEST_QUERY = "What are signs of rejection after transplant?"
LESSON_TEST_TOPIC = "When to call the transplant team"
QUIZ_TEST_TOPIC = "Signs of rejection after transplant"

TOP_K = 3


# Step 6: Dense Retriever Utilities

This section defines helper functions for transformer-based retrieval. Dense retrievers convert text into numerical vectors called embeddings. FAISS then searches these vectors to find chunks that are closest to the query vector.

The notebook supports two pooling methods. Mean pooling averages token embeddings across the input text. CLS pooling uses the first model token representation. Different retriever models use different pooling conventions, so both options are included.


In [ ]:
# Utility functions for dense transformer retrievers.

def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


def cls_pool(last_hidden_state):
    return last_hidden_state[:, 0]


def encode_texts(texts, tokenizer, model, pooling="mean", batch_size=16, max_length=512, normalize=True):
    model.eval()
    vectors = []

    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            batch = texts[start:start + batch_size]
            encoded = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt",
            ).to(DEVICE)

            output = model(**encoded)

            if pooling == "cls":
                batch_vectors = cls_pool(output.last_hidden_state)
            else:
                batch_vectors = mean_pool(output.last_hidden_state, encoded["attention_mask"])

            if normalize:
                batch_vectors = torch.nn.functional.normalize(batch_vectors, p=2, dim=1)

            vectors.append(batch_vectors.cpu().numpy().astype("float32"))

    return np.vstack(vectors)


def build_ip_faiss_index(embeddings):
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings.astype("float32"))
    return index


def dense_search(query, retriever, top_k=5):
    query_vector = encode_texts(
        [query],
        retriever["query_tokenizer"],
        retriever["query_model"],
        pooling=retriever["query_pooling"],
        batch_size=1,
        normalize=True,
    )
    scores, indices = retriever["index"].search(query_vector, top_k)
    return indices[0].tolist(), scores[0].tolist()

# Step 7: BM25 Retriever

BM25 is a lexical retrieval method. Lexical retrieval means that it matches query words against document words. BM25 is simple, fast, and useful when the manual contains the same terms as the user question.

This notebook uses the `rank_bm25` Python package. The package implements BM25Okapi, which is a common BM25 variant. Documentation is available at [https://pypi.org/project/rank-bm25/](https://pypi.org/project/rank-bm25/).


In [ ]:
# Build the BM25 retriever.
# BM25 is the lexical retriever family used in MedRAG.

def tokenize_for_bm25(text):
    return re.findall(r"[a-z0-9]+", text.lower())


bm25_tokenized_chunks = [tokenize_for_bm25(chunk) for chunk in chunks]
bm25 = BM25Okapi(bm25_tokenized_chunks)

bm25_export = {
    "retriever": "BM25",
    "tokenized_chunks": bm25_tokenized_chunks,
    "note": "Use rank_bm25.BM25Okapi(tokenized_chunks) to reload this lexical retriever.",
}
(INDEX_PATH / "bm25_index.json").write_text(json.dumps(bm25_export), encoding="utf-8")

def bm25_search(query, top_k=5):
    tokenized_query = tokenize_for_bm25(query)
    scores = bm25.get_scores(tokenized_query)
    indices = np.argsort(scores)[::-1][:top_k]
    return indices.tolist(), scores[indices].tolist()

print("BM25 ready")

BM25 ready


# Step 8: Contriever Retriever

Contriever is a dense retriever trained for unsupervised information retrieval. It can retrieve semantically related text even when the query and document do not use the exact same words.

This notebook uses the public `facebook/contriever` model from Hugging Face. The related paper is available at [https://arxiv.org/abs/2112.09118](https://arxiv.org/abs/2112.09118).


In [ ]:
# Build the Contriever retriever.
# Contriever is a general semantic retriever used in MedRAG.

retrievers = {}

contriever_model_name = "facebook/contriever"
contriever_tokenizer = AutoTokenizer.from_pretrained(contriever_model_name)
contriever_model = AutoModel.from_pretrained(contriever_model_name).to(DEVICE)

start_time = time.time()
contriever_embeddings = encode_texts(
    chunks,
    contriever_tokenizer,
    contriever_model,
    pooling="mean",
    batch_size=16,
    normalize=True,
)
contriever_index = build_ip_faiss_index(contriever_embeddings)

np.save(INDEX_PATH / "contriever_embeddings.npy", contriever_embeddings)
faiss.write_index(contriever_index, str(INDEX_PATH / "contriever_faiss.index"))

retrievers["contriever"] = {
    "name": "contriever",
    "model_name": contriever_model_name,
    "query_tokenizer": contriever_tokenizer,
    "query_model": contriever_model,
    "query_pooling": "mean",
    "index": contriever_index,
}

print("Contriever ready in seconds:", round(time.time() - start_time, 2))
print("Embedding shape:", contriever_embeddings.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

BertModel LOAD REPORT from: facebook/contriever
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Contriever ready in seconds: 93.53
Embedding shape: (175, 768)


# Step 9: SPECTER Retriever

SPECTER is a dense model designed for scientific document representation. It was originally developed to represent scientific papers using citation-informed training.

This notebook tests SPECTER because MedRAG-style medical retrieval often compares general, scientific, and biomedical retrieval models. The related SPECTER paper is available at [https://arxiv.org/abs/2004.07180](https://arxiv.org/abs/2004.07180).


In [ ]:
# Build the SPECTER retriever.
# SPECTER is a scientific-document semantic retriever used in MedRAG.

specter_model_name = "allenai/specter"
specter_tokenizer = AutoTokenizer.from_pretrained(specter_model_name)
specter_model = AutoModel.from_pretrained(specter_model_name).to(DEVICE)

start_time = time.time()
specter_embeddings = encode_texts(
    chunks,
    specter_tokenizer,
    specter_model,
    pooling="cls",
    batch_size=16,
    normalize=True,
)
specter_index = build_ip_faiss_index(specter_embeddings)

np.save(INDEX_PATH / "specter_embeddings.npy", specter_embeddings)
faiss.write_index(specter_index, str(INDEX_PATH / "specter_faiss.index"))

retrievers["specter"] = {
    "name": "specter",
    "model_name": specter_model_name,
    "query_tokenizer": specter_tokenizer,
    "query_model": specter_model,
    "query_pooling": "cls",
    "index": specter_index,
}

print("SPECTER ready in seconds:", round(time.time() - start_time, 2))
print("Embedding shape:", specter_embeddings.shape)

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: allenai/specter
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


SPECTER ready in seconds: 107.18
Embedding shape: (175, 768)


# Step 10: MedCPT Retriever

MedCPT is a biomedical dense retrieval model. It uses separate encoders for queries and articles. This design is useful because questions and medical documents often have different wording.

This notebook tests MedCPT because it is medically specialized and relevant to medical RAG retrieval. The MedCPT publication is indexed at PubMed: [https://pubmed.ncbi.nlm.nih.gov/37971130/](https://pubmed.ncbi.nlm.nih.gov/37971130/).


In [ ]:
# Build the MedCPT retriever.
# MedCPT is the biomedical dense retriever used in MedRAG.
# It uses one encoder for queries and a separate encoder for document chunks.

medcpt_query_model_name = "ncbi/MedCPT-Query-Encoder"
medcpt_article_model_name = "ncbi/MedCPT-Article-Encoder"

medcpt_query_tokenizer = AutoTokenizer.from_pretrained(medcpt_query_model_name)
medcpt_query_model = AutoModel.from_pretrained(medcpt_query_model_name).to(DEVICE)

medcpt_article_tokenizer = AutoTokenizer.from_pretrained(medcpt_article_model_name)
medcpt_article_model = AutoModel.from_pretrained(medcpt_article_model_name).to(DEVICE)

start_time = time.time()
medcpt_embeddings = encode_texts(
    chunks,
    medcpt_article_tokenizer,
    medcpt_article_model,
    pooling="cls",
    batch_size=16,
    normalize=True,
)
medcpt_index = build_ip_faiss_index(medcpt_embeddings)

np.save(INDEX_PATH / "medcpt_embeddings.npy", medcpt_embeddings)
faiss.write_index(medcpt_index, str(INDEX_PATH / "medcpt_faiss.index"))

retrievers["medcpt"] = {
    "name": "medcpt",
    "model_name": medcpt_article_model_name,
    "query_model_name": medcpt_query_model_name,
    "query_tokenizer": medcpt_query_tokenizer,
    "query_model": medcpt_query_model,
    "query_pooling": "cls",
    "index": medcpt_index,
}

print("MedCPT ready in seconds:", round(time.time() - start_time, 2))
print("Embedding shape:", medcpt_embeddings.shape)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MedCPT ready in seconds: 83.13
Embedding shape: (175, 768)


# Step 11: Retriever Evaluation

This section compares BM25, Contriever, SPECTER, and MedCPT on the same four questions. For each question, each retriever returns the top three chunks.

The output table includes the query, retriever name, rank, chunk index, retrieval score, and retrieved chunk text. The retrieval score is only directly comparable within the same retriever because each method calculates scores differently.

The CSV file is saved to Drive so the retrieval results can be reviewed later and included in the project report.


In [ ]:
# Evaluate all four retrievers on four questions.
# Each retriever returns only the top 3 chunks.
# This creates a compact table for retrieval comparison.

def search_with_retriever(retriever_name, query, top_k=3):
    if retriever_name == "bm25":
        return bm25_search(query, top_k=top_k)
    return dense_search(query, retrievers[retriever_name], top_k=top_k)


all_retriever_names = ["bm25", "contriever", "specter", "medcpt"]
rows = []

for query in EVAL_QUERIES:
    for retriever_name in all_retriever_names:
        indices, scores = search_with_retriever(retriever_name, query, top_k=TOP_K)

        for rank, (idx, score) in enumerate(zip(indices, scores), start=1):
            rows.append({
                "query": query,
                "retriever": retriever_name,
                "rank": rank,
                "chunk_index": idx,
                "score": float(score),
                "chunk_text": chunks[idx],
            })

retrieval_eval_df = pd.DataFrame(rows)

retrieval_eval_df.to_csv(
    REPORT_PATH / "retrieval_eval_results_small.csv",
    index=False
)

display(retrieval_eval_df)

print("Rows:", len(retrieval_eval_df))
print("Saved:", REPORT_PATH / "retrieval_eval_results_small.csv")

,query,retriever,rank,chunk_index,score,chunk_text
0,What temperature is considered dangerous after...,bm25,1,17,6.078402,"Heart rate, also called pulse When checking bl..."
1,What temperature is considered dangerous after...,bm25,2,6,5.834460,Vital sign Safe range Weight Blood pressure He...
2,What temperature is considered dangerous after...,bm25,3,5,5.834460,If any of your child’s vital signs are not in ...
3,What temperature is considered dangerous after...,contriever,1,139,0.570776,Common side effects • Sores in the mouth • Hig...
4,What temperature is considered dangerous after...,contriever,2,40,0.560768,Follow the best practices in this section to h...
5,What temperature is considered dangerous after...,contriever,3,83,0.545495,"12 months after transplant, your child can tra..."
6,What temperature is considered dangerous after...,specter,1,172,0.850367,Call the transplant team if your child has:  ...
7,What temperature is considered dangerous after...,specter,2,169,0.845405,Call the transplant team if your child has:  ...
8,What temperature is considered dangerous after...,specter,3,168,0.839976,P002100 2-weeks of Daily Vital Signs Date Weig...
9,What temperature is considered dangerous after...,medcpt,1,92,0.689727,Signs of rejection Many signs of rejection can...


Rows: 48
Saved: /content/drive/MyDrive/high_risk_project/medrag_retriever_outputs/reports/retrieval_eval_results_small.csv


# Step 12: Retrieval Results and Final Retriever Decision

The evaluation compared four retrievers across four questions. Each retriever returned the top three chunks for each question. The results show that no single retriever was strongest on every type of query. At the same time, the notebook results also show that some retrievers retrieved at least one useful chunk more often than the earlier summary suggested.

---

## Relevant Chunk Coverage Across the Four Questions

The table below separates two ideas:

1. whether a retriever returned at least one relevant chunk in its top three
2. whether that retriever returned the strongest or most exact result for the question

| Question | BM25 | Contriever | SPECTER | MedCPT |
|---|---|---|---|---|
| What temperature is considered dangerous after transplant? | Yes (2/3 relevant; useful temperature warning chunks, but not the exact 101.4°F threshold) | No (0/3 clearly relevant) | Yes (3/3 relevant; strongest exact threshold retrieval) | Yes (3/3 relevant, but the chunks centered on 100.4°F rejection warnings instead of the exact 101.4°F threshold) |
| When should I call the transplant team? | Yes (3/3 relevant; one strong direct chunk and two narrower call-the-team situations) | Yes (3/3 relevant, but scattered and less direct) | Yes (3/3 relevant, but only one chunk gave the main urgent contact pathway clearly) | Yes (2/3 relevant; narrower situations, less direct than BM25) |
| What are signs of rejection after transplant? | Yes (3/3 relevant; strongest direct retrieval) | Yes (3/3 relevant) | Yes (2/3 relevant) | Yes (2/3 relevant) |
| My child feels hot, is tired, and seems different than usual. What should I do? | Yes (1/3 weakly relevant; the top two chunks drifted to diabetes) | No (0/3 clearly relevant) | No (0/3 clearly relevant) | Yes (1/3 strongly relevant; best match for indirect symptom phrasing) |

This table shows that "best retriever" and "retrieved at least one relevant chunk" are not always the same conclusion. For example, BM25 was not the best retriever for the temperature question, but it still retrieved two useful temperature-related chunks in its top three. MedCPT was not the best retriever for the exact threshold question either, but it still returned temperature-related warning chunks. The strongest retriever was decided by the precision and usefulness of the top results, not only by whether one relevant chunk appeared somewhere in the top three.

---

## BM25

**Main observation:** BM25 had the broadest useful coverage across the four questions, but it was strongest when the query matched the wording of the manual directly.

**Advantages**

- Retrieved at least one relevant chunk for all four questions.
- Retrieved two useful temperature-related chunks for the temperature question, even though it missed the exact 101.4°F / 38.2°C threshold.
- Was strongest on the direct manual-style questions, especially the rejection question.
- Retrieved three relevant chunks for the transplant-team question, including one direct urgent-contact chunk.
- Was easy to interpret because the ranking followed word overlap with the manual.

**Disadvantages**

- Did not retrieve the most exact answer for the temperature question.
- Performed poorly on the indirect symptom phrasing question.
- The top two chunks for the indirect question drifted to medicine-induced diabetes because the query included the word "tired."
- Its useful coverage was broader than the earlier summary suggested, but its precision dropped when caregiver wording did not match the manual closely.

---

## Contriever

**Main observation:** Contriever worked as a semantic baseline and retrieved some relevant clinical content, but it did not show a unique retrieval strength.

**Advantages**

- Retrieved three relevant chunks for the rejection question.
- Retrieved several relevant call-the-team situations for the transplant-team question.
- Showed that semantic retrieval could recover the rejection section even without exact word overlap.
- Provided a useful comparison point against BM25 and the medically specialized dense retrievers.

**Disadvantages**

- Retrieved no clearly relevant chunk for the temperature question.
- Retrieved no clearly relevant chunk for the indirect symptom question.
- Missed the strongest results on the temperature, transplant-team, and indirect symptom questions.
- Did not provide a clear unique reason to include it in the final hybrid design.

---

## SPECTER

**Main observation:** SPECTER was the strongest retriever for the exact temperature-threshold question, but it was less reliable for indirect caregiver phrasing.

**Advantages**

- Was the strongest retriever for the temperature question.
- Retrieved three relevant chunks for that question.
- The first two chunks directly included the exact threshold of 101.4°F / 38.2°C.
- Retrieved two strong chunks for the rejection question.
- Showed that dense retrieval could recover a specific clinical threshold very precisely.

**Disadvantages**

- Was less consistent for the transplant-team question.
- Only one retrieved chunk clearly captured the main after-hours and emergency instructions.
- Retrieved no clearly relevant chunk for the indirect symptom question.
- Did not provide the broadest coverage across the four questions.

---

## MedCPT

**Main observation:** MedCPT was the strongest retriever for the indirect symptom phrasing question and was the most useful model when the caregiver wording did not match the manual directly.

**Advantages**

- Retrieved at least one relevant chunk for all four questions.
- Was the strongest retriever for the indirect symptom question.
- Returned the rejection warning-sign chunk as the top result for the indirect caregiver-style query.
- Retrieved relevant chunks for the temperature and rejection questions.
- Added a medically specialized retrieval strength that the other models did not match as well on indirect phrasing.

**Disadvantages**

- Did not retrieve the most exact answer for the temperature question.
- Its temperature-related chunks focused on 100.4°F / 38°C rejection warnings instead of the exact 101.4°F / 38.2°C threshold.
- Was less direct than BM25 on the rejection question.
- Retrieved narrower and less central call-the-team chunks than BM25 for the transplant-team question.

---

## Final Retriever Selection

**Main observation:** The reevaluated results still support the same hybrid choice, but they show more clearly that BM25 had broader useful coverage than the earlier summary gave it credit for.

**Why BM25 was kept**

- Retrieved at least one relevant chunk for all four questions.
- Was strongest on the direct manual-style questions.
- Added broad useful coverage even when it was not the most exact retriever.

**Why SPECTER was kept**

- Was the strongest retriever for the exact temperature-threshold question.
- Retrieved the most precise numeric answer in the evaluation.

**Why MedCPT was kept**

- Was the strongest retriever for the indirect symptom phrasing question.
- Added a medically specialized strength for caregiver-style language.

**Why Contriever was excluded**

- Retrieved useful chunks for some questions, especially the rejection question.
- Did not provide a unique retrieval win on any of the four questions.
- Did not improve the final hybrid design enough to justify inclusion.

**Final decision**

- The hybrid strategy combines BM25, SPECTER, and MedCPT.
- This selection prioritizes retrieval accuracy and safety over backend simplicity.
- The planned hybrid method retrieves top chunks from BM25, SPECTER, and MedCPT, removes duplicates, and combines rankings using reciprocal rank fusion.



# Step 13: Reload Hybrid Retrieval Artifacts

This section reloads the saved retrieval files from Google Drive. It does not rebuild the document embeddings from the manual.

The reload step uses the saved BM25 tokens, the saved SPECTER FAISS index, and the saved MedCPT FAISS index. It still loads the SPECTER and MedCPT query encoders because dense retrievers need an encoder to convert a new user question into a query vector.

This design supports rerunning the generation section without rerunning the full embedding and indexing process.


In [4]:
# Reload saved retrieval artifacts for hybrid retrieval.
# This section avoids rebuilding document embeddings from the manual.

SELECTED_RETRIEVERS = ["bm25", "specter", "medcpt"]
HYBRID_TOP_K_PER_RETRIEVER = 3
HYBRID_MAX_CONTEXT_CHUNKS = 6
RRF_K = 60

with open(DATA_PATH / "chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

with open(INDEX_PATH / "bm25_index.json", "r", encoding="utf-8") as f:
    bm25_export = json.load(f)

bm25 = BM25Okapi(bm25_export["tokenized_chunks"])

specter_model_name = "allenai/specter"
specter_tokenizer = AutoTokenizer.from_pretrained(specter_model_name)
specter_model = AutoModel.from_pretrained(specter_model_name).to(DEVICE)
specter_index = faiss.read_index(str(INDEX_PATH / "specter_faiss.index"))

medcpt_query_model_name = "ncbi/MedCPT-Query-Encoder"
medcpt_query_tokenizer = AutoTokenizer.from_pretrained(medcpt_query_model_name)
medcpt_query_model = AutoModel.from_pretrained(medcpt_query_model_name).to(DEVICE)
medcpt_index = faiss.read_index(str(INDEX_PATH / "medcpt_faiss.index"))

hybrid_retrievers = {
    "bm25": {
        "name": "bm25",
        "type": "lexical",
        "index": bm25,
    },
    "specter": {
        "name": "specter",
        "type": "dense",
        "query_tokenizer": specter_tokenizer,
        "query_model": specter_model,
        "query_pooling": "cls",
        "index": specter_index,
        "model_name": specter_model_name,
    },
    "medcpt": {
        "name": "medcpt",
        "type": "dense",
        "query_tokenizer": medcpt_query_tokenizer,
        "query_model": medcpt_query_model,
        "query_pooling": "cls",
        "index": medcpt_index,
        "query_model_name": medcpt_query_model_name,
        "article_model_name": "ncbi/MedCPT-Article-Encoder",
    },
}

print("Loaded chunks:", len(chunks))
print("Loaded hybrid retrievers:", SELECTED_RETRIEVERS)
print("SPECTER index vectors:", specter_index.ntotal)
print("MedCPT index vectors:", medcpt_index.ntotal)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded chunks: 175
Loaded hybrid retrievers: ['bm25', 'specter', 'medcpt']
SPECTER index vectors: 175
MedCPT index vectors: 175


In [5]:
# Configure Groq.
# The API key is loaded from Colab Secrets or entered through a hidden prompt.

from getpass import getpass

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = None

if not GROQ_API_KEY:
    GROQ_API_KEY = getpass("Enter GROQ_API_KEY: ")

GROQ_MODEL = "llama-3.3-70b-versatile"
client = Groq(api_key=GROQ_API_KEY)

print("Groq client ready")


Enter GROQ_API_KEY: ··········
Groq client ready


# Step 14: Hybrid Retrieval and JSON Generation

This section defines the hybrid retrieval function and the generation prompt format.

The hybrid retrieval function gets top chunks from BM25, SPECTER, and MedCPT. Then it removes duplicate chunks and ranks the merged results using reciprocal rank fusion. This avoids comparing raw BM25, SPECTER, and MedCPT scores directly because each retriever uses a different scoring scale.

The prompt format supports three backend tasks: question answering, lesson generation, and quiz generation. Each prompt includes retrieved context chunks with citation numbers. The model is instructed to use only the provided context and return valid JSON.


In [6]:
# Build hybrid retrieval, prompts, and JSON generation.

def tokenize_for_bm25(text):
    return re.findall(r"[a-z0-9]+", text.lower())


def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


def cls_pool(last_hidden_state):
    return last_hidden_state[:, 0]


def encode_query(text, tokenizer, model, pooling="mean", max_length=512, normalize=True):
    model.eval()
    with torch.no_grad():
        encoded = tokenizer(
            [text],
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        ).to(DEVICE)

        output = model(**encoded)

        if pooling == "cls":
            vector = cls_pool(output.last_hidden_state)
        else:
            vector = mean_pool(output.last_hidden_state, encoded["attention_mask"])

        if normalize:
            vector = torch.nn.functional.normalize(vector, p=2, dim=1)

        return vector.cpu().numpy().astype("float32")


def search_bm25(query, top_k=3):
    tokenized_query = tokenize_for_bm25(query)
    scores = bm25.get_scores(tokenized_query)
    indices = np.argsort(scores)[::-1][:top_k]
    return indices.tolist(), scores[indices].tolist()


def search_dense(query, retriever_name, top_k=3):
    retriever = hybrid_retrievers[retriever_name]
    query_vector = encode_query(
        query,
        retriever["query_tokenizer"],
        retriever["query_model"],
        pooling=retriever["query_pooling"],
        normalize=True,
    )
    scores, indices = retriever["index"].search(query_vector, top_k)
    return indices[0].tolist(), scores[0].tolist()


def hybrid_retrieve_context(
    query,
    retriever_names=SELECTED_RETRIEVERS,
    top_k_per_retriever=HYBRID_TOP_K_PER_RETRIEVER,
    max_context_chunks=HYBRID_MAX_CONTEXT_CHUNKS,
):
    merged = {}

    for retriever_name in retriever_names:
        if retriever_name == "bm25":
            indices, scores = search_bm25(query, top_k=top_k_per_retriever)
        else:
            indices, scores = search_dense(query, retriever_name, top_k=top_k_per_retriever)

        for rank, (idx, score) in enumerate(zip(indices, scores), start=1):
            if idx not in merged:
                merged[idx] = {
                    "index": idx,
                    "text": chunks[idx],
                    "rrf_score": 0.0,
                    "retrievers": [],
                    "raw_scores": {},
                }

            merged[idx]["rrf_score"] += 1 / (RRF_K + rank)
            merged[idx]["retrievers"].append({"name": retriever_name, "rank": rank})
            merged[idx]["raw_scores"][retriever_name] = float(score)

    ranked_items = sorted(
        merged.values(),
        key=lambda item: (item["rrf_score"], len(item["retrievers"])),
        reverse=True,
    )

    return ranked_items[:max_context_chunks]


def build_prompt(input_text, context_items, task):
    indexed_context = "\n".join(
        [f"[{i}] {item['text']}" for i, item in enumerate(context_items)]
    )

    if task == "qa":
        instruction = "Answer the question using only the provided context."
        schema = """
{
  "answer": "string markdown with citations like [0]",
  "key_points": ["string", "string"],
  "source_indices": [0, 1],
  "sources": ["exact source text", "exact source text"]
}
""".strip()

    elif task == "lesson":
        instruction = "Create a clear lesson for a parent based only on the provided context."
        schema = """
{
  "title": "string",
  "lesson_markdown": "string markdown with citations like [0]",
  "key_takeaways": ["string", "string"],
  "source_indices": [0, 1],
  "sources": ["exact source text", "exact source text"]
}
""".strip()

    elif task == "quiz":
        instruction = "Create a short quiz based only on the provided context."
        schema = """
{
  "questions": [
    {
      "question": "string",
      "type": "multiple_choice | true_false",
      "options": ["string", "string"],
      "answer": "string",
      "explanation": "string with citations like [0]"
    }
  ],
  "source_indices": [0, 1],
  "sources": ["exact source text", "exact source text"]
}
""".strip()

    else:
        raise ValueError("Unsupported task")

    return f"""
You are a medical assistant helping parents understand a transplant care manual.

Return only valid JSON.
Use only the provided context.
Do not add external medical knowledge.
Use citation markers like [0], [1], and [2].
The source_indices field must match the citations used.
The sources field must contain the exact raw text for the cited context items.

Context chunks:
{indexed_context}

Task:
{instruction}

Input:
{input_text}

Return JSON with this schema:
{schema}
""".strip()


def generate_json_with_groq(prompt):
    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
        max_tokens=1600,
    )

    raw = response.choices[0].message.content or ""

    start = raw.find("{")
    end = raw.rfind("}")

    if start == -1 or end == -1:
        raise ValueError("No JSON object found in model response")

    return json.loads(raw[start:end + 1])


# Step 15: Ask Generation Test

This section tests the planned question-answering backend format. It retrieves context with the hybrid retriever and sends that context to Groq.

The test question asks about signs of rejection after transplant. This question is used because the retrieval evaluation showed strong rejection-related retrieval, while the hybrid retriever can also include support from multiple retrieval methods.

This test checks whether the notebook can retrieve chunks, build a prompt, call Groq, parse JSON, and save the result in the planned `/ask` response format.


In [9]:
# Generate one test answer for the /ask backend task using hybrid retrieval.

ask_context = hybrid_retrieve_context(ASK_TEST_QUERY)
ask_prompt = build_prompt(ASK_TEST_QUERY, ask_context, "qa")
ask_result = generate_json_with_groq(ask_prompt)

(REPORT_PATH / "ask_test_result_hybrid.json").write_text(
    json.dumps(ask_result, indent=2),
    encoding="utf-8"
)

print("Retrievers:", SELECTED_RETRIEVERS)
print("Context chunk indices:", [item["index"] for item in ask_context])
print(json.dumps(ask_result, indent=2))


Retrievers: ['bm25', 'specter', 'medcpt']
Context chunk indices: [92, 91, 93, 35, 67]
{
  "answer": "Signs of rejection after transplant include temperature of 100.4\u2070F or 38\u2070C or higher, shortness of breath, swelling in the hands or feet, nausea, vomiting, or bloating, fast or uneven heart rate, or pulse, lower blood pressure, and new aches or pains [0], [2]. Many signs of rejection cannot be seen because they happen inside the cells of the body [0], [1].",
  "key_points": [
    "Temperature of 100.4\u2070F or 38\u2070C or higher",
    "Shortness of breath, swelling, nausea, vomiting, bloating, fast or uneven heart rate, lower blood pressure, and new aches or pains"
  ],
  "source_indices": [
    0,
    1,
    2
  ],
  "sources": [
    "Signs of rejection Many signs of rejection cannot be seen because they happen inside the cells of the body. If you see any of these signs of possible rejection, use the table below to call the transplant team immediately: \u2022 Temperature of

# Step 16: Lesson Generation Test

This section tests the planned lesson-generation backend format. The lesson task is useful for the iOS application because it turns manual content into a structured educational explanation for parents.

The output includes a title, markdown lesson text, key takeaways, source indices, and source text. These fields match the planned structure for the backend and app.


In [10]:
# Generate one test lesson for the /lesson backend task using hybrid retrieval.

lesson_context = hybrid_retrieve_context(LESSON_TEST_TOPIC)
lesson_prompt = build_prompt(LESSON_TEST_TOPIC, lesson_context, "lesson")
lesson_result = generate_json_with_groq(lesson_prompt)

(REPORT_PATH / "lesson_test_result_hybrid.json").write_text(
    json.dumps(lesson_result, indent=2),
    encoding="utf-8"
)

print("Retrievers:", SELECTED_RETRIEVERS)
print("Context chunk indices:", [item["index"] for item in lesson_context])
print(json.dumps(lesson_result, indent=2))


Retrievers: ['bm25', 'specter', 'medcpt']
Context chunk indices: [4, 8, 83, 63, 7, 3]
{
  "title": "When to Call the Transplant Team",
  "lesson_markdown": "If any of your child\u2019s vital signs are not in the safe range, call the transplant team [0]. You can also call the transplant team if you are concerned about your child\u2019s emotions or mental wellness [3]. For non-urgent messages, you can use MyChart to send a message to the transplant team and they will reply within 2 business days with an answer or a referral [0]. If you need emergency help, call the hospital operator at (650) 497-8000 or call 911 [4].",
  "key_takeaways": [
    "Call the transplant team if your child's vital signs are not in the safe range",
    "Use MyChart for non-urgent messages and call the hospital operator or 911 for emergencies"
  ],
  "source_indices": [
    0,
    3,
    4
  ],
  "sources": [
    "They will reply within 2 business days with an answer or a referral. To use MyChart go to https:/ /m

# Step 17: Quiz Generation Test

This section tests the planned quiz-generation backend format. The quiz task checks whether the system can use retrieved manual chunks to create questions, answers, and explanations.

The quiz output is structured as JSON. Each question includes an explanation with a citation. This keeps the quiz grounded in the manual instead of general model knowledge.


In [11]:
# Generate one test quiz for the /quiz backend task using hybrid retrieval.

quiz_context = hybrid_retrieve_context(QUIZ_TEST_TOPIC)
quiz_prompt = build_prompt(QUIZ_TEST_TOPIC, quiz_context, "quiz")
quiz_result = generate_json_with_groq(quiz_prompt)

(REPORT_PATH / "quiz_test_result_hybrid.json").write_text(
    json.dumps(quiz_result, indent=2),
    encoding="utf-8"
)

print("Retrievers:", SELECTED_RETRIEVERS)
print("Context chunk indices:", [item["index"] for item in quiz_context])
print(json.dumps(quiz_result, indent=2))


Retrievers: ['bm25', 'specter', 'medcpt']
Context chunk indices: [91, 92, 93, 90, 67]
{
  "questions": [
    {
      "question": "What is the body's normal response to anything it wasn't born with?",
      "type": "multiple_choice",
      "options": [
        "Acceptance",
        "Rejection"
      ],
      "answer": "Rejection",
      "explanation": "Rejection is the body\u2019s normal response to anything it wasn\u2019t born with [0]."
    },
    {
      "question": "What is a common sign of possible rejection?",
      "type": "multiple_choice",
      "options": [
        "Fever below 100.4\u2070F",
        "Temperature of 100.4\u2070F or 38\u2070C or higher"
      ],
      "answer": "Temperature of 100.4\u2070F or 38\u2070C or higher",
      "explanation": "If you see any of these signs of possible rejection, use the table below to call the transplant team immediately: \u2022 Temperature of 100.4\u2070F or 38\u2070C or higher [1]."
    },
    {
      "question": "Is rejection most c

# Step 18: Backend Service Output Analysis

This section evaluates the generated outputs from the three planned backend-style services: question answering, lesson generation, and quiz generation. The goal is to check whether the hybrid retrieval and Groq generation pipeline produced accurate, grounded, and usable responses in the planned backend response format.

The hybrid retriever used BM25, SPECTER, and MedCPT. The retrieved context chunks were then passed to Groq for structured JSON generation. The analysis focuses on factual accuracy, citation quality, completeness, and usefulness for a parent-facing transplant education application.

---

## Ask Output Analysis

The `/ask` test used the question:

`What are signs of rejection after transplant?`

The retrieved context chunk indices were:

`[92, 91, 93, 35, 67]`

The generated answer was accurate and well grounded. It correctly listed the main signs of possible rejection from the manual:

- Temperature of `100.4°F / 38°C` or higher
- Shortness of breath
- Swelling in the hands or feet
- Nausea, vomiting, or bloating
- Fast or uneven heart rate or pulse
- Lower blood pressure
- New aches or pains

The answer also included the important clarification that many signs of rejection cannot be seen because they happen inside the cells of the body. This statement is directly supported by the retrieved manual text.

The citation quality was strong. The answer cited chunks `[0]`, `[1]`, and `[2]`, and these chunks directly supported the response. The cited sources matched the retrieved context and included the exact rejection warning-sign language.

The output was also concise. It did not add external medical knowledge, and it did not include unsupported recommendations. This is important because the planned backend is intended to answer using only the transplant manual.

One limitation is that the answer did not explicitly say to call the transplant team immediately if these signs are present. The source text includes that instruction, but the generated answer focused mainly on listing signs. For a healthcare education system, including the action step would make the answer safer and more complete.

Overall, the `/ask` output was accurate, grounded, and appropriate. A minor improvement would be to include the manual’s action instruction when warning signs are listed.

---

## Lesson Output Analysis

The `/lesson` test used the topic:

`When to call the transplant team`

The retrieved context chunk indices were:

`[4, 8, 83, 63, 7, 3]`

The generated lesson was mostly accurate and useful. It correctly included several important situations where a parent should contact the transplant team:

- If vital signs are not in the safe range
- If there are concerns about emotional or mental wellness
- If the child is not feeling well before a catheterization procedure
- If the situation is an emergency

The lesson also correctly separated non-urgent and urgent communication. It mentioned MyChart for non-urgent messages and emergency support options for urgent situations. This is useful because parents need to know which communication method is appropriate for different levels of concern.

The citation quality was acceptable. The lesson cited chunks `[0]`, `[3]`, and `[4]`, and each cited chunk supported the relevant claim. The source text for chunk `[0]` supported vital signs and MyChart. Chunk `[3]` supported mental wellness concerns and illness before catheterization. Chunk `[4]` supported after-hours contact and emergency help.

However, the lesson was less complete than it could be. It did not include the full list of symptoms from the manual, such as fever, body aches, headache, coughing, sore throat, nausea, vomiting, stomach pain, diarrhea, mouth blisters, swelling, or weight gain. These symptoms are important because they give parents concrete warning signs.

The lesson also included emergency guidance, but it could be clearer. It referred to emergency support, while the manual separates after-hours paging from immediate ambulance emergencies. A clearer output would distinguish between calling the hospital operator for after-hours transplant support and calling 911 for immediate emergency help.

Overall, the `/lesson` output was accurate and grounded, but moderately incomplete. It covered the general decision points well, but it missed some specific symptom examples that would make the lesson more useful.

---

## Quiz Output Analysis

The `/quiz` test used the topic:

`Signs of rejection after transplant`

The retrieved context chunk indices were:

`[91, 92, 93, 90, 67]`

The generated quiz was accurate and clearly grounded in the manual. It created questions about rejection, including definition, timing, and common signs.

Each question was supported by the retrieved manual chunks. The first and third questions used chunk `[0]`, which explains what rejection is and when it is most common. The sign-related question used chunk `[1]`, which lists signs of possible rejection.

The citation quality was strong. Each explanation included a citation, and the cited source text directly supported the answer. The quiz did not include unsupported medical claims.

The quiz was also appropriate for parent education. The questions were short, simple, and focused on important concepts. The answer choices were clear and not overly complex.

One limitation is that the quiz did not test the full range of rejection signs. It only tested one example sign. The manual also lists shortness of breath, swelling, nausea, vomiting, bloating, fast or uneven heart rate, lower blood pressure, and new aches or pains. Including one question about these additional symptoms would make the quiz more complete.

Another limitation is that the quiz did not ask what action to take if signs of rejection appear. The manual states that parents should call the transplant team immediately. Since this is an important safety behavior, a stronger quiz would include a question about contacting the transplant team.

Overall, the `/quiz` output was accurate, grounded, and easy to understand. It was useful as a short quiz, but it could be improved by covering more symptoms and including the action step.

---

## Overall Output Quality

The three planned backend-style outputs show that the hybrid retrieval and generation pipeline works. The outputs were grounded in retrieved manual chunks, used citations, and returned structured JSON that matches the planned backend format.

The strongest output was the `/ask` response. It directly answered the question and listed the relevant signs of rejection. The quiz output was also strong because it produced simple, accurate questions with cited explanations. The lesson output was accurate but less complete because it missed several specific symptoms listed in the manual.

The main strength of the hybrid approach is that it provided relevant context for different backend tasks. The rejection-related tasks retrieved highly relevant chunks, and the generated outputs stayed close to those chunks.

The main limitation is completeness. The generation step sometimes summarized the retrieved context instead of including all clinically relevant details. This matters because parent-facing healthcare education should include clear warning signs and clear action steps.

A future improvement would be to strengthen the prompt instructions for safety-related content. For example, when the retrieved context includes warning signs and an action instruction, the model should include both the warning signs and the required action, such as calling the transplant team immediately.

Overall, the backend-style outputs produced accurate and grounded results. The system is appropriate as a manual-based educational RAG prototype, but safety-related outputs should be reviewed for completeness before later integration.


# Step 19: Backend Export

This section exports the planned hybrid backend package to Google Drive. The package contains the retrieval files required to reconstruct BM25, SPECTER, and MedCPT retrieval for a later backend integration step.

The exported files include `chunks.json`, `bm25_index.json`, `specter_faiss.index`, `medcpt_faiss.index`, and `hybrid_retriever_config.json`. The FAISS files store the dense indexes that were already built earlier in the notebook. The config file records which retrievers are included and how many chunks are retrieved from each retriever.


In [12]:
# Export a hybrid backend package.

import shutil

BACKEND_PACKAGE_PATH = OUTPUT_PATH / "backend_package_hybrid"
BACKEND_PACKAGE_PATH.mkdir(exist_ok=True)

(BACKEND_PACKAGE_PATH / "chunks.json").write_text(
    json.dumps(chunks, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

(BACKEND_PACKAGE_PATH / "bm25_index.json").write_text(
    json.dumps(bm25_export),
    encoding="utf-8"
)

shutil.copy2(INDEX_PATH / "specter_faiss.index", BACKEND_PACKAGE_PATH / "specter_faiss.index")
shutil.copy2(INDEX_PATH / "medcpt_faiss.index", BACKEND_PACKAGE_PATH / "medcpt_faiss.index")

hybrid_config = {
    "selected_retrievers": SELECTED_RETRIEVERS,
    "top_k_per_retriever": HYBRID_TOP_K_PER_RETRIEVER,
    "max_context_chunks": HYBRID_MAX_CONTEXT_CHUNKS,
    "rrf_k": RRF_K,
    "dense_retrievers": {
        "specter": {
            "model_name": "allenai/specter",
            "index_file": "specter_faiss.index",
            "pooling": "cls",
        },
        "medcpt": {
            "query_model_name": "ncbi/MedCPT-Query-Encoder",
            "article_model_name": "ncbi/MedCPT-Article-Encoder",
            "index_file": "medcpt_faiss.index",
            "pooling": "cls",
        },
    },
    "lexical_retrievers": {
        "bm25": {
            "index_file": "bm25_index.json",
        }
    },
}

(BACKEND_PACKAGE_PATH / "hybrid_retriever_config.json").write_text(
    json.dumps(hybrid_config, indent=2),
    encoding="utf-8"
)

print("Backend package created:", BACKEND_PACKAGE_PATH)
print("Package files:")

for file_path in sorted(BACKEND_PACKAGE_PATH.iterdir()):
    print("-", file_path.name)


Backend package created: /content/drive/MyDrive/high_risk_project/medrag_retriever_outputs/backend_package_hybrid
Package files:
- bm25_index.json
- chunks.json
- hybrid_retriever_config.json
- medcpt_faiss.index
- specter_faiss.index


# Step 20: Backend Package Reload Test

This section verifies that the exported hybrid backend package can be reloaded. This is a smoke test, which means it checks the basic function of the exported files without testing the entire backend server.

The test reloads the chunk file, BM25 file, SPECTER FAISS index, MedCPT FAISS index, and hybrid config file. Then it runs one hybrid retrieval query about rejection signs. The saved output should show that the package files reload correctly and that the hybrid retriever returns relevant manual chunks.


In [13]:
# Verify that the exported hybrid backend package can be reloaded.

with open(BACKEND_PACKAGE_PATH / "chunks.json", "r", encoding="utf-8") as f:
    backend_chunks = json.load(f)

with open(BACKEND_PACKAGE_PATH / "bm25_index.json", "r", encoding="utf-8") as f:
    backend_bm25_data = json.load(f)

with open(BACKEND_PACKAGE_PATH / "hybrid_retriever_config.json", "r", encoding="utf-8") as f:
    backend_hybrid_config = json.load(f)

backend_bm25 = BM25Okapi(backend_bm25_data["tokenized_chunks"])
backend_specter_index = faiss.read_index(str(BACKEND_PACKAGE_PATH / "specter_faiss.index"))
backend_medcpt_index = faiss.read_index(str(BACKEND_PACKAGE_PATH / "medcpt_faiss.index"))

package_hybrid_retrievers = dict(hybrid_retrievers)
package_hybrid_retrievers["bm25"] = {
    "name": "bm25",
    "type": "lexical",
    "index": backend_bm25,
}
package_hybrid_retrievers["specter"] = {
    **package_hybrid_retrievers["specter"],
    "index": backend_specter_index,
}
package_hybrid_retrievers["medcpt"] = {
    **package_hybrid_retrievers["medcpt"],
    "index": backend_medcpt_index,
}

original_hybrid_retrievers = hybrid_retrievers
hybrid_retrievers = package_hybrid_retrievers

backend_smoke_context = hybrid_retrieve_context(
    "What are signs of rejection after transplant?",
    retriever_names=backend_hybrid_config["selected_retrievers"],
    top_k_per_retriever=backend_hybrid_config["top_k_per_retriever"],
    max_context_chunks=backend_hybrid_config["max_context_chunks"],
)

hybrid_retrievers = original_hybrid_retrievers

backend_smoke_test = pd.DataFrame({
    "rank": list(range(1, len(backend_smoke_context) + 1)),
    "chunk_index": [item["index"] for item in backend_smoke_context],
    "rrf_score": [item["rrf_score"] for item in backend_smoke_context],
    "retrievers": [item["retrievers"] for item in backend_smoke_context],
    "chunk_text": [item["text"] for item in backend_smoke_context],
})

display(backend_smoke_test)

print("Reloaded backend chunks:", len(backend_chunks))
print("Reloaded BM25 tokenized chunks:", len(backend_bm25_data["tokenized_chunks"]))
print("Reloaded SPECTER index vectors:", backend_specter_index.ntotal)
print("Reloaded MedCPT index vectors:", backend_medcpt_index.ntotal)
print("Reloaded hybrid config retrievers:", backend_hybrid_config["selected_retrievers"])


,rank,chunk_index,rrf_score,retrievers,chunk_text
0,1,92,0.048916,"[{'name': 'bm25', 'rank': 1}, {'name': 'specte...",Signs of rejection Many signs of rejection can...
1,2,91,0.048652,"[{'name': 'bm25', 'rank': 2}, {'name': 'specte...",Rejection is the body’s normal response to any...
2,3,93,0.015873,"[{'name': 'bm25', 'rank': 3}]",If you see any of these signs of possible reje...
3,4,35,0.015873,"[{'name': 'specter', 'rank': 3}]","The biggest problems after transplant, even re..."
4,5,67,0.015873,"[{'name': 'medcpt', 'rank': 3}]",1 year after transplant When your child reache...


Reloaded backend chunks: 175
Reloaded BM25 tokenized chunks: 175
Reloaded SPECTER index vectors: 175
Reloaded MedCPT index vectors: 175
Reloaded hybrid config retrievers: ['bm25', 'specter', 'medcpt']


# Conclusion

This notebook used MedRAG as a retrieval comparison reference. It tested BM25, Contriever, SPECTER, and MedCPT on the transplant manual rather than on external medical corpora.

The retrieval evaluation supported a hybrid approach using BM25, SPECTER, and MedCPT. This choice prioritized retrieval accuracy for healthcare education. BM25 performed well on direct manual-style questions, SPECTER performed best on the temperature threshold question, and MedCPT performed best on the indirect symptom question.

The backend-style generation tests showed that the planned response format can produce accurate, grounded, and structured outputs. The strongest results appeared in the rejection-related tasks, while the lesson output showed that completeness remains an important area for improvement.

The main limitation is that hybrid retrieval requires more backend resources than BM25 alone. However, this notebook represents an initial project step, and the class focus prioritizes accuracy and safety over deployment simplicity. A later implementation step can integrate the hybrid retriever into the backend and refine prompting for safety-critical completeness.

Overall, this notebook establishes the retrieval and output design for the next project stage. The planned system is a manual-grounded RAG pipeline in which a hybrid retriever supplies context, Groq generates structured JSON, and the backend returns those responses to the application.
